# Preparacion para clustering

Este notebook prepara una matriz numerica inicial para clustering usando variables del PDF. Tambien genera una vista PCA de dos componentes para revisar si hay estructura antes de probar algoritmos como K-Means, Gaussian Mixture, DBSCAN o HDBSCAN.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from feature_config import CLUSTERING_FEATURE_SETS, LOG_CANDIDATE_COLUMNS, RADIUS_BINS, RADIUS_LABELS
from eda_exodata import find_csv, load_data

pio.renderers.default = 'notebook_connected'
csv_path = find_csv(None)
df = load_data(csv_path)
display(Markdown(f'**Dataset:** `{csv_path.name}` con `{df.shape[0]}` filas y `{df.shape[1]}` columnas.'))

**Dataset:** `PSCompPars_2026.04.25_17.36.36.csv` con `6273` filas y `86` columnas.

## Seleccion de features

El set inicial recomendado conserva buena cobertura y evita duplicar unidades (`pl_rade` vs `pl_radj`, `pl_bmasse` vs `pl_bmassj`).

In [2]:
feature_set_name = 'pdf_core_with_mass_density'
features = CLUSTERING_FEATURE_SETS[feature_set_name]
features = [col for col in features if col in df.columns]

coverage = df[features].notna().all(axis=1).mean() * 100
features_inline = '`, `'.join(features)
message = (
    f"**Feature set:** `{feature_set_name}`  \n"
    f"**Features:** `{features_inline}`  \n"
    f"**Filas completas:** `{coverage:.2f}%`"
)
display(Markdown(message))

df[features].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

**Feature set:** `pdf_core_with_mass_density`  
**Features:** `pl_rade`, `pl_bmasse`, `pl_dens`, `pl_orbper`, `pl_orbsmax`, `pl_orbeccen`, `st_teff`, `st_met`, `sy_pnum`  
**Filas completas:** `79.12%`

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
pl_rade,6223.0,5.806406,5.403930e+00,0.309800,0.750898,1.073000,1.840000,2.841418,11.900000,14.300000,18.250942,8.720587e+01
pl_bmasse,6242.0,400.731995,1.135293e+03,0.020000,0.383870,1.330500,4.270000,9.200000,184.023339,2511.448289,6463.274850,9.534852e+03
pl_dens,6199.0,13.109384,2.600833e+02,0.030255,0.184310,0.341966,1.337925,2.588308,4.672604,11.304261,52.975966,1.388095e+04
pl_orbper,5934.0,72130.776525,5.221021e+06,0.090706,0.645625,1.552174,4.302426,10.678822,38.050250,1535.041692,13647.773972,4.020000e+08
pl_orbsmax,5845.0,15.694857,3.487157e+02,0.004400,0.013166,0.024720,0.052300,0.102000,0.307000,4.071200,67.560000,1.900000e+04
pl_orbeccen,5221.0,0.079098,1.528254e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.090000,0.410000,0.730000,9.500000e-01
st_teff,5979.0,5392.978659,1.733535e+03,415.000000,3099.780000,3515.000000,4895.985000,5543.000000,5897.000000,6340.000000,7257.660000,5.700000e+04
st_met,5633.0,0.016542,1.883061e-01,-1.000000,-0.540000,-0.306000,-0.080000,0.020000,0.130000,0.320000,0.414680,7.900000e-01
sy_pnum,6273.0,1.759126,1.149467e+00,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,4.000000,6.000000,8.000000e+00


## Transformaciones

Las variables positivas y muy sesgadas se transforman con `log10`. Despues se imputan nulos con mediana y se escala todo con `StandardScaler`.

In [3]:
log_features = [col for col in features if col in LOG_CANDIDATE_COLUMNS]
linear_features = [col for col in features if col not in log_features]

def safe_log10(values):
    frame = pd.DataFrame(values, columns=log_features).copy()
    frame = frame.where(frame > 0)
    return np.log10(frame)

preprocess = ColumnTransformer(
    transformers=[
        ('log', Pipeline([
            ('log10', FunctionTransformer(safe_log10, validate=False, feature_names_out='one-to-one')),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), log_features),
        ('linear', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), linear_features),
    ],
    remainder='drop',
)

X = preprocess.fit_transform(df[features])
transformed_feature_names = [f'log10_{col}' for col in log_features] + linear_features
X_df = pd.DataFrame(X, columns=transformed_feature_names, index=df.index)
X_df.head()

,log10_pl_rade,log10_pl_bmasse,log10_pl_dens,log10_pl_orbper,log10_pl_orbsmax,pl_orbeccen,st_teff,st_met,sy_pnum
0,1.264731,2.212381,1.659603,1.432662,1.189396,1.208127,-0.310774,-1.551848,-0.660468
1,1.273547,2.192500,1.594120,1.661361,1.345034,0.099413,-0.701302,-0.206776,-0.660468
2,1.341598,1.603246,0.138527,1.164768,0.940143,-0.461961,-0.302503,-1.271625,-0.660468
3,1.290966,1.983302,1.094280,2.262217,1.713032,2.122466,-0.036636,2.315235,0.209569
4,1.374080,1.315952,-0.569407,1.874416,1.393580,4.309721,0.206779,0.241582,-0.660468


In [4]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_df)
pca_df = pd.DataFrame(coords, columns=['PC1', 'PC2'])
pca_df['pl_name'] = df['pl_name'].values
pca_df['hostname'] = df['hostname'].values
pca_df['discoverymethod'] = df['discoverymethod'].values
pca_df['pl_rade'] = df['pl_rade'].values
pca_df['radius_class'] = pd.cut(df['pl_rade'], bins=RADIUS_BINS, labels=RADIUS_LABELS).astype(str)

display(Markdown(
    f'**Varianza explicada:** PC1 `{pca.explained_variance_ratio_[0]:.2%}`, '
    f'PC2 `{pca.explained_variance_ratio_[1]:.2%}`'
))
pca_df.head()

**Varianza explicada:** PC1 `35.17%`, PC2 `16.69%`

,PC1,PC2,pl_name,hostname,discoverymethod,pl_rade,radius_class
0,2.799369,1.922262,11 Com b,11 Com,Radial Velocity,12.2,Gigante gaseoso (>10)
1,2.709110,1.385447,11 UMi b,11 UMi,Radial Velocity,12.3,Gigante gaseoso (>10)
2,2.063304,0.487688,14 And b,14 And,Radial Velocity,13.1,Gigante gaseoso (>10)
3,3.984334,1.262146,14 Her b,14 Her,Radial Velocity,12.5,Gigante gaseoso (>10)
4,4.306558,0.937116,16 Cyg B b,16 Cyg B,Radial Velocity,13.5,Gigante gaseoso (>10)


In [5]:
fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='discoverymethod',
    symbol='radius_class',
    hover_name='pl_name',
    hover_data=['hostname', 'pl_rade'],
    title='PCA exploratorio de variables preparadas para clustering',
    height=650,
)
fig.show()

## Matriz lista para modelado

La tabla `X_df` queda imputada, transformada y escalada. Es la entrada natural para probar clustering. Mantener `df[['pl_name', 'hostname', 'discoverymethod']]` separado ayuda a interpretar los grupos sin meter identificadores al modelo.

In [6]:
model_ready = pd.concat(
    [df[['pl_name', 'hostname', 'discoverymethod']].reset_index(drop=True), X_df.reset_index(drop=True)],
    axis=1,
)
model_ready.head()

,pl_name,hostname,discoverymethod,log10_pl_rade,log10_pl_bmasse,log10_pl_dens,log10_pl_orbper,log10_pl_orbsmax,pl_orbeccen,st_teff,st_met,sy_pnum
0,11 Com b,11 Com,Radial Velocity,1.264731,2.212381,1.659603,1.432662,1.189396,1.208127,-0.310774,-1.551848,-0.660468
1,11 UMi b,11 UMi,Radial Velocity,1.273547,2.192500,1.594120,1.661361,1.345034,0.099413,-0.701302,-0.206776,-0.660468
2,14 And b,14 And,Radial Velocity,1.341598,1.603246,0.138527,1.164768,0.940143,-0.461961,-0.302503,-1.271625,-0.660468
3,14 Her b,14 Her,Radial Velocity,1.290966,1.983302,1.094280,2.262217,1.713032,2.122466,-0.036636,2.315235,0.209569
4,16 Cyg B b,16 Cyg B,Radial Velocity,1.374080,1.315952,-0.569407,1.874416,1.393580,4.309721,0.206779,0.241582,-0.660468
